In [64]:
import pandas as pd
import sqlite3 



df = pd.read_csv("amazon.csv")  
df.columns

Index(['product_id', 'product_name', 'category', 'discounted_price',
       'actual_price', 'discount_percentage', 'rating', 'rating_count',
       'about_product', 'user_id', 'user_name', 'review_id', 'review_title',
       'review_content', 'img_link', 'product_link'],
      dtype='str')

In [2]:
df = pd.read_csv("amazon.csv")
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
conn = sqlite3.connect('AmazonDB.db')
df.to_sql('Productos',conn, if_exists='replace', index=False)


1465

In [61]:
consulta = ("SELECT category, AVG(rating) as avg_rating, COUNT(*) as count FROM productos GROUP BY category")
conn = sqlite3.connect('AmazonDB.db')
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  avg_rating  count
0          Car&Motorbike    3.800000      1
1  Computers&Accessories    4.154967    453
2            Electronics    4.081749    526
3    Health&PersonalCare    4.000000      1
4           Home&Kitchen    4.031696    448
5        HomeImprovement    4.250000      2
6     MusicalInstruments    3.900000      2
7         OfficeProducts    4.309677     31
8             Toys&Games    4.300000      1


In [65]:
conn = sqlite3.connect("AmazonDB.db")
df["category"]= df["category"].str.split('|').str[0]
print(df.columns)
df[["product_id","category"]].to_sql("temp_productos",conn, if_exists="replace", index=False)

Index(['product_id', 'product_name', 'category', 'discounted_price',
       'actual_price', 'discount_percentage', 'rating', 'rating_count',
       'about_product', 'user_id', 'user_name', 'review_id', 'review_title',
       'review_content', 'img_link', 'product_link'],
      dtype='str')


1465

In [66]:
cursor = conn.cursor()
cursor.execute("""
UPDATE Productos  
SET category = (
SELECT temp_productos.category
FROM temp_productos
WHERE temp_productos.product_id = Productos.product_id
)
WHERE EXISTS(
SELECT 1
FROM temp_productos
WHERE temp_productos.product_id = Productos.product_id
)
""")

In [5]:
conn.commit()
cursor.execute("DROP TABLE temp_productos;")
conn.close()

In [26]:
consulta = ("SELECT product_id, product_name, category, MAX(CAST(REPLACE(rating_count, ',','') AS INT)) as rating_count FROM Productos  GROUP BY category")
conn = sqlite3.connect("AmazonDB.db")
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

   product_id                                       product_name  \
0  B0912WJ87V  Reffair AX30 [MAX] Portable Air Purifier for C...   
1  B005FYNT3G          SanDisk Cruzer Blade 32GB USB Flash Drive   
2  B07KSMBL2H  AmazonBasics Flexible Premium HDMI Cable (Blac...   
3  B07BKSSDR2  Dr Trust Electronic Kitchen Digital Scale Weig...   
4  B01LWYDEQ7  Pigeon Polypropylene Mini Handy and Compact Ch...   
5  B07R99NBVB  Gizga Essentials Cable Organiser, Cord Managem...   
6  B076B8G5D8  Boya ByM1 Auxiliary Omnidirectional Lavalier C...   
7  B00AXHBBXU  Casio FX-82MS 2nd Gen Non-Programmable Scienti...   
8  B00DJ5N9VK  Faber-Castell Connector Pen Set - Pack of 25 (...   

                category  rating_count  
0          Car&Motorbike          1118  
1  Computers&Accessories        253105  
2            Electronics        426973  
3    Health&PersonalCare          3663  
4           Home&Kitchen        270563  
5        HomeImprovement          5985  
6     MusicalInstruments        

In [18]:
consulta = ("SELECT category, COUNT(*), AVG(CAST(rating as FLOAT)) as promedio FROM Productos WHERE rating != '|' GROUP BY category")
conn = sqlite3.connect("AmazonDB.db")
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  COUNT(*)  promedio
0          Car&Motorbike         1  3.800000
1  Computers&Accessories       453  4.154967
2            Electronics       526  4.081749
3    Health&PersonalCare         1  4.000000
4           Home&Kitchen       447  4.040716
5        HomeImprovement         2  4.250000
6     MusicalInstruments         2  3.900000
7         OfficeProducts        31  4.309677
8             Toys&Games         1  4.300000


In [4]:
consulta = ("SELECT category, COUNT(*), AVG(CAST(REPLACE(rating_count, ',' , '') as INT)) as Promedio FROM Productos GROUP BY category")
conn = sqlite3.connect("AmazonDB.db")
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  COUNT(*)      Promedio
0          Car&Motorbike         1   1118.000000
1  Computers&Accessories       453  17136.782705
2            Electronics       526  29997.809886
3    Health&PersonalCare         1   3663.000000
4           Home&Kitchen       448   6676.493304
5        HomeImprovement         2   4283.000000
6     MusicalInstruments         2  44441.000000
7         OfficeProducts        31   4828.225806
8             Toys&Games         1  15867.000000


In [17]:
consulta = ("SELECT category,COUNT(*), AVG(REPLACE(REPLACE(discounted_price, '₹', ''),',','')) AS d_price, AVG(REPLACE(REPLACE(actual_price, '₹', ''),',','')) AS a_price, AVG(REPLACE(discount_percentage, '%', '')) AS d_percentage  FROM Productos GROUP BY category")
conn = sqlite3.connect("AmazonDB.db")
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  COUNT(*)      d_price       a_price  d_percentage
0          Car&Motorbike         1  2339.000000   4000.000000     42.000000
1  Computers&Accessories       453   842.650375   1683.623135     54.024283
2            Electronics       526  5965.887833  10127.311787     50.828897
3    Health&PersonalCare         1   899.000000   1900.000000     53.000000
4           Home&Kitchen       448  2330.615647   4162.073661     40.120536
5        HomeImprovement         2   337.000000    799.000000     57.500000
6     MusicalInstruments         2   638.000000   1347.000000     46.000000
7         OfficeProducts        31   301.580645    397.193548     12.354839
8             Toys&Games         1   150.000000    150.000000      0.000000


In [67]:
rating = pd.to_numeric(df["rating"], errors="coerce")
discounted_price =  pd.to_numeric(df["discounted_price"].str.replace('₹','').str.replace(',',''), errors='coerce')
print(rating.corr(discounted_price))




0.12033741712288533


In [54]:
consulta = ("SELECT category, COUNT(*) as count, AVG(CAST(rating as FLOAT)) AS rating_avg FROM Productos WHERE rating != '|' GROUP BY category "
"HAVING COUNT(*) > 5 ORDER BY rating DESC" )
conn = sqlite3.connect("AmazonDB.db")
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  count  rating_avg
0            Electronics    526    4.081749
1         OfficeProducts     31    4.309677
2  Computers&Accessories    453    4.154967
3           Home&Kitchen    447    4.040716


In [55]:

df = pd.read_csv("amazon.csv") 
df.dtypes

product_id             str
product_name           str
category               str
discounted_price       str
actual_price           str
discount_percentage    str
rating                 str
rating_count           str
about_product          str
user_id                str
user_name              str
review_id              str
review_title           str
review_content         str
img_link               str
product_link           str
dtype: object

0         ₹399
1         ₹199
2         ₹199
3         ₹329
4         ₹154
         ...  
1460      ₹379
1461    ₹2,280
1462    ₹2,219
1463    ₹1,399
1464    ₹2,863
Name: discounted_price, Length: 1465, dtype: str